## Variáveis de Ambiente

- Este notebook foi feito para ser importado pelos jobs do pipeline MUNIC
- O objetivo é definir as variáveis comuns aos notebooks do pipeline, facilitando a gestão delas e centralizando a definição em um único lugar, evitando erros de digitação ao longo do pipeline

In [0]:
catalog_name = 'munic_catalog'
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'

Atualmente usa-se o munic_file_paths somente na ingest_to_landing, nos demais usa-se o referece_year como base para selecionar a tabela a ser processada no pipeline 

In [0]:
landing_folder_path = '/Volumes/munic_catalog/landing/raw_files'
bronze_folder_path = '/Volumes/munic_catalog/bronze/csv_files'

# Parametrização para múltiplos anos
reference_year = 2024  # Alterar para o ano desejado

# URL dos arquivos de origem. E define quais anos devem ser puxados  
munic_source_url_map = {
    2024: "https://ftp.ibge.gov.br/Perfil_Municipios/2024/Base_de_Dados/Base_MUNIC_2024_20251107.xlsx",
    2019: "https://ftp.ibge.gov.br/Perfil_Municipios/2019/Base_de_Dados/Base_MUNIC_2019_20210817.xlsx",
    # Adicionar outros anos conforme a necessidade
}

# Caminho do arquivo de cada edição na landing (derivado da URL, sem duplicação manual)
munic_file_paths = {
    year: f"{landing_folder_path}/{url.split('/')[-1]}"
    for year, url in munic_source_url_map.items()
}

# Variáveis da edição corrente (reference_year), consumidas por landing_to_bronze e bronze_to_silver.
# Este bloco só escolhe QUAL arquivo processar. A detecção de abas/colunas (célula abaixo)
# não lê nada daqui, ela só recebe os nomes do XLSX da edição selecionada na váriavle reference_year.
munic_source_url = munic_source_url_map[reference_year]
munic_file_name  = munic_source_url.split("/")[-1]
munic_file       = f"{landing_folder_path}/{munic_file_name}"


In [0]:
# --- Overrides por edição (escape opcional) -----------------------------------
# A LÓGICA de detecção (detect_munic_sheets / detect_identity_renames) está no
# env_functions; aqui ficam apenas os overrides por edição, chaveados por reference_year.
# No dia a dia ficam VAZIOS — preencha apenas a edição problemática.
# São consultados pelo landing_to_bronze antes de chamar a detecção automática.
munic_sheet_overrides = {}            # ex.: {2019: {"dict": "Dicionário", "info": "Comunicção e informática", "gov": "Governança"}}
munic_identity_rename_overrides = {}  # ex.: {2019: {"codmun": "cod_munic", "nome_munic": "desc_mun", "classe_pop": "faixa_populacao"}}